In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [4]:
df = pd.read_csv("garments_worker_productivity.csv")

df.columns = df.columns.str.strip().str.lower()

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Dataset shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [5]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nNumerical Summary:")
display(df.describe())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64

date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


Duplicate Rows: 0

Numerical Summary:


,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
count,1197.000000,1197.000000,1197.000000,691.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000
mean,6.426901,0.729632,15.062172,1190.465991,4567.460317,38.210526,0.730159,0.369256,0.150376,34.609858,0.735091
std,3.463963,0.097891,10.943219,1837.455001,3348.823563,160.182643,12.709757,3.268987,0.427848,22.197687,0.174488
min,1.000000,0.070000,2.900000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.233705
25%,3.000000,0.700000,3.940000,774.500000,1440.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.650307
50%,6.000000,0.750000,15.260000,1039.000000,3960.000000,0.000000,0.000000,0.000000,0.000000,34.000000,0.773333
75%,9.000000,0.800000,24.260000,1252.500000,6960.000000,50.000000,0.000000,0.000000,0.000000,57.000000,0.850253
max,12.000000,0.800000,54.560000,23122.000000,25920.000000,3600.000000,300.000000,45.000000,2.000000,89.000000,1.120437


In [6]:
data = df.copy()

if "date" in data.columns:
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["month"] = data["date"].dt.month
    data["day_of_week"] = data["date"].dt.dayofweek
    data.drop(columns=["date"], inplace=True)

data["MeetsTarget"] = (
    data["actual_productivity"] >= data["targeted_productivity"]
).astype(int)

y_reg = data["actual_productivity"].copy()
y_cls = data["MeetsTarget"].copy()

X_reg = data.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

X_cls = data.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

print("Regression features shape:", X_reg.shape)
print("Classification features shape:", X_cls.shape)

print("\nRegression target:")
print(y_reg.head())

print("\nClassification target distribution:")
display(y_cls.value_counts())

Regression features shape: (1197, 15)
Classification features shape: (1197, 15)

Regression target:
0    0.940725
1    0.886500
2    0.800570
3    0.800570
4    0.800382
Name: actual_productivity, dtype: float64

Classification target distribution:


MeetsTarget
1    875
0    322
Name: count, dtype: int64

In [7]:
indices = np.arange(len(data))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

X_train_reg = X_reg.iloc[train_indices]
X_test_reg = X_reg.iloc[test_indices]

y_train_reg = y_reg.iloc[train_indices]
y_test_reg = y_reg.iloc[test_indices]

X_train_cls = X_cls.iloc[train_indices]
X_test_cls = X_cls.iloc[test_indices]

y_train_cls = y_cls.iloc[train_indices]
y_test_cls = y_cls.iloc[test_indices]

print("Training rows:", len(train_indices))
print("Testing rows:", len(test_indices))

print("\nRegression train:", X_train_reg.shape)
print("Regression test:", X_test_reg.shape)

print("\nClassification train:", X_train_cls.shape)
print("Classification test:", X_test_cls.shape)

Training rows: 957
Testing rows: 240

Regression train: (957, 15)
Regression test: (240, 15)

Classification train: (957, 15)
Classification test: (240, 15)


In [8]:
numeric_features = X_train_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train_reg.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("\nPreprocessing pipeline created successfully.")

Numerical features: ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical features: ['quarter', 'department', 'day']

Preprocessing pipeline created successfully.


C:\Users\Hirav\AppData\Local\Temp\ipykernel_4568\2478039638.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train_reg.select_dtypes(


In [9]:
regression_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

start_train = time.perf_counter()

regression_model.fit(X_train_reg, y_train_reg)

regression_train_time = time.perf_counter() - start_train

start_predict = time.perf_counter()

y_pred_reg = regression_model.predict(X_test_reg)

regression_prediction_time = time.perf_counter() - start_predict

print("Linear Regression training completed.")
print(f"Training time: {regression_train_time:.6f} seconds")
print(f"Prediction time: {regression_prediction_time:.6f} seconds")

Linear Regression training completed.
Training time: 0.086498 seconds
Prediction time: 0.008395 seconds


In [10]:
regression_mae = mean_absolute_error(
    y_test_reg, y_pred_reg
)

regression_rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_reg)
)

regression_r2 = r2_score(
    y_test_reg, y_pred_reg
)

regression_results = {
    "Model": "Linear Regression (Scikit-learn)",
    "MAE": regression_mae,
    "RMSE": regression_rmse,
    "R2": regression_r2,
    "Training Time (s)": regression_train_time,
    "Prediction Time (s)": regression_prediction_time
}

display(pd.DataFrame([regression_results]))

display(pd.DataFrame({
    "Actual Productivity": y_test_reg.iloc[:10].values,
    "Predicted Productivity": y_pred_reg[:10]
}))

,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Linear Regression (Scikit-learn),0.108453,0.148618,0.168168,0.086498,0.008395


,Actual Productivity,Predicted Productivity
0,0.268214,0.623913
1,0.800359,0.770490
2,0.681061,0.828393
3,0.325000,0.735027
4,0.667604,0.816348
5,0.800980,0.797048
6,0.768847,0.550193
7,0.768847,0.584974
8,0.650417,0.696046
9,0.750396,0.751620


In [11]:
classification_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

start_train = time.perf_counter()

classification_model.fit(X_train_cls, y_train_cls)

classification_train_time = time.perf_counter() - start_train

start_predict = time.perf_counter()

y_pred_cls = classification_model.predict(X_test_cls)

classification_prediction_time = time.perf_counter() - start_predict

print("Logistic Regression training completed.")
print(f"Training time: {classification_train_time:.6f} seconds")
print(f"Prediction time: {classification_prediction_time:.6f} seconds")

Logistic Regression training completed.
Training time: 0.042075 seconds
Prediction time: 0.011689 seconds


In [12]:
classification_accuracy = accuracy_score(
    y_test_cls, y_pred_cls
)

classification_precision = precision_score(
    y_test_cls, y_pred_cls, zero_division=0
)

classification_recall = recall_score(
    y_test_cls, y_pred_cls, zero_division=0
)

classification_f1 = f1_score(
    y_test_cls, y_pred_cls, zero_division=0
)

classification_results = {
    "Model": "Logistic Regression (Scikit-learn)",
    "Accuracy": classification_accuracy,
    "Precision": classification_precision,
    "Recall": classification_recall,
    "F1-Score": classification_f1,
    "Training Time (s)": classification_train_time,
    "Prediction Time (s)": classification_prediction_time
}

display(pd.DataFrame([classification_results]))

display(pd.DataFrame({
    "Actual MeetsTarget": y_test_cls.iloc[:10].values,
    "Predicted MeetsTarget": y_pred_cls[:10]
}))

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Logistic Regression (Scikit-learn),0.775,0.780822,0.966102,0.863636,0.042075,0.011689


,Actual MeetsTarget,Predicted MeetsTarget
0,0,1
1,1,1
2,0,1
3,0,1
4,0,1
5,1,1
6,1,0
7,1,1
8,1,1
9,1,1


In [13]:
print("PART A COMPLETED")
print("- Linear Regression trained and evaluated.")
print("- Logistic Regression trained and evaluated.")
print("- One fixed train-test split used for both tasks.")
print("- Preprocessing includes imputation, encoding, and scaling.")
print("- Training and prediction times recorded.")

PART A COMPLETED
- Linear Regression trained and evaluated.
- Logistic Regression trained and evaluated.
- One fixed train-test split used for both tasks.
- Preprocessing includes imputation, encoding, and scaling.
- Training and prediction times recorded.


In [14]:
import numpy as np
import pandas as pd
import time

In [15]:
def manual_preprocess(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

    for col in numeric_cols:
        median_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_value)
        X_test[col] = X_test[col].fillna(median_value)

    for col in categorical_cols:
        mode_value = X_train[col].mode().iloc[0]
        X_train[col] = X_train[col].fillna(mode_value)
        X_test[col] = X_test[col].fillna(mode_value)

    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, dtype=int)
    X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, dtype=int)

    X_test_encoded = X_test_encoded.reindex(
        columns=X_train_encoded.columns,
        fill_value=0
    )

    means = X_train_encoded.mean()
    stds = X_train_encoded.std(ddof=0).replace(0, 1)

    X_train_scaled = (X_train_encoded - means) / stds
    X_test_scaled = (X_test_encoded - means) / stds

    return (
        X_train_scaled.to_numpy(dtype=float),
        X_test_scaled.to_numpy(dtype=float)
    )

In [17]:
X_train_manual, X_test_manual = manual_preprocess(
    X_reg.iloc[train_indices],
    X_reg.iloc[test_indices]
)

y_train_reg_manual = y_reg.iloc[train_indices].to_numpy(dtype=float)
y_test_reg_manual = y_reg.iloc[test_indices].to_numpy(dtype=float)

y_train_cls_manual = y_cls.iloc[train_indices].to_numpy(dtype=float)
y_test_cls_manual = y_cls.iloc[test_indices].to_numpy(dtype=float)

print("Training shape:", X_train_manual.shape)
print("Testing shape:", X_test_manual.shape)
print("Regression target shape:", y_train_reg_manual.shape)
print("Classification target shape:", y_train_cls_manual.shape)

Training shape: (957, 26)
Testing shape: (240, 26)
Regression target shape: (957,)
Classification target shape: (957,)


In [18]:
def manual_linear_regression(X_train, y_train, X_test):
    X_train_bias = np.c_[np.ones(X_train.shape[0]), X_train]
    X_test_bias = np.c_[np.ones(X_test.shape[0]), X_test]

    coefficients = np.linalg.lstsq(X_train_bias, y_train, rcond=None)[0]

    y_pred_train = X_train_bias @ coefficients
    y_pred_test = X_test_bias @ coefficients

    return y_pred_train, y_pred_test, coefficients

In [19]:
start_time = time.perf_counter()

y_train_pred_reg_manual, y_pred_reg_manual, reg_coefficients = manual_linear_regression(
    X_train_manual,
    y_train_reg_manual,
    X_test_manual
)

reg_train_time_manual = time.perf_counter() - start_time

start_time = time.perf_counter()

X_test_bias = np.c_[np.ones(X_test_manual.shape[0]), X_test_manual]
y_pred_reg_manual = X_test_bias @ reg_coefficients

reg_prediction_time_manual = time.perf_counter() - start_time

print("Manual Linear Regression Training Time:", reg_train_time_manual)
print("Manual Linear Regression Prediction Time:", reg_prediction_time_manual)

Manual Linear Regression Training Time: 0.009658800001488999
Manual Linear Regression Prediction Time: 0.0005441000103019178


In [20]:
mae_reg_manual = np.mean(np.abs(y_test_reg_manual - y_pred_reg_manual))
rmse_reg_manual = np.sqrt(np.mean((y_test_reg_manual - y_pred_reg_manual) ** 2))

ss_res = np.sum((y_test_reg_manual - y_pred_reg_manual) ** 2)
ss_tot = np.sum((y_test_reg_manual - np.mean(y_test_reg_manual)) ** 2)

r2_reg_manual = 1 - (ss_res / ss_tot)

reg_manual_results = pd.DataFrame({
    "Model": ["Linear Regression (Manual)"],
    "MAE": [mae_reg_manual],
    "RMSE": [rmse_reg_manual],
    "R2": [r2_reg_manual],
    "Training Time (s)": [reg_train_time_manual],
    "Prediction Time (s)": [reg_prediction_time_manual]
})

display(reg_manual_results)

,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Linear Regression (Manual),0.108457,0.148628,0.168047,0.009659,0.000544


In [21]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

In [22]:
def manual_logistic_regression(X_train, y_train, X_test, learning_rate=0.1, iterations=3000, l2=0.01):
    X_train_bias = np.c_[np.ones(X_train.shape[0]), X_train]
    X_test_bias = np.c_[np.ones(X_test.shape[0]), X_test]

    weights = np.zeros(X_train_bias.shape[1])

    for _ in range(iterations):
        probabilities = sigmoid(X_train_bias @ weights)
        gradient = (X_train_bias.T @ (probabilities - y_train)) / len(y_train)
        gradient[1:] += (l2 / len(y_train)) * weights[1:]
        weights -= learning_rate * gradient

    train_probabilities = sigmoid(X_train_bias @ weights)
    test_probabilities = sigmoid(X_test_bias @ weights)

    y_train_pred = (train_probabilities >= 0.5).astype(int)
    y_test_pred = (test_probabilities >= 0.5).astype(int)

    return y_train_pred, y_test_pred, test_probabilities, weights

In [23]:
start_time = time.perf_counter()

y_train_pred_cls_manual, y_pred_cls_manual, y_prob_cls_manual, cls_weights_manual = manual_logistic_regression(
    X_train_manual,
    y_train_cls_manual,
    X_test_manual
)

cls_train_time_manual = time.perf_counter() - start_time

start_time = time.perf_counter()

X_test_bias_cls = np.c_[np.ones(X_test_manual.shape[0]), X_test_manual]
y_prob_cls_manual = sigmoid(X_test_bias_cls @ cls_weights_manual)
y_pred_cls_manual = (y_prob_cls_manual >= 0.5).astype(int)

cls_prediction_time_manual = time.perf_counter() - start_time

print("Manual Logistic Regression Training Time:", cls_train_time_manual)
print("Manual Logistic Regression Prediction Time:", cls_prediction_time_manual)

Manual Logistic Regression Training Time: 0.12034279998624697
Manual Logistic Regression Prediction Time: 0.00043689997983165085


In [24]:
tp = np.sum((y_test_cls_manual == 1) & (y_pred_cls_manual == 1))
tn = np.sum((y_test_cls_manual == 0) & (y_pred_cls_manual == 0))
fp = np.sum((y_test_cls_manual == 0) & (y_pred_cls_manual == 1))
fn = np.sum((y_test_cls_manual == 1) & (y_pred_cls_manual == 0))

accuracy_cls_manual = (tp + tn) / len(y_test_cls_manual)
precision_cls_manual = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_cls_manual = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_cls_manual = (
    2 * precision_cls_manual * recall_cls_manual /
    (precision_cls_manual + recall_cls_manual)
    if (precision_cls_manual + recall_cls_manual) > 0 else 0
)

cls_manual_results = pd.DataFrame({
    "Model": ["Logistic Regression (Manual)"],
    "Accuracy": [accuracy_cls_manual],
    "Precision": [precision_cls_manual],
    "Recall": [recall_cls_manual],
    "F1-Score": [f1_cls_manual],
    "Training Time (s)": [cls_train_time_manual],
    "Prediction Time (s)": [cls_prediction_time_manual]
})

display(cls_manual_results)

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Logistic Regression (Manual),0.770833,0.779817,0.960452,0.860759,0.120343,0.000437


In [25]:
display(pd.DataFrame({
    "Actual MeetsTarget": y_test_cls_manual[:10].astype(int),
    "Predicted MeetsTarget": y_pred_cls_manual[:10]
}))

,Actual MeetsTarget,Predicted MeetsTarget
0,0,1
1,1,1
2,0,1
3,0,1
4,0,1
5,1,1
6,1,0
7,1,1
8,1,1
9,1,1


In [28]:
regression_comparison = pd.concat(
    [
        pd.DataFrame([regression_results]),
        reg_manual_results
    ],
    ignore_index=True
)

display(regression_comparison)

,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s)
0,Linear Regression (Scikit-learn),0.108453,0.148618,0.168168,0.086498,0.008395
1,Linear Regression (Manual),0.108457,0.148628,0.168047,0.009659,0.000544


In [29]:
tp_sklearn = np.sum((y_test_cls == 1) & (y_pred_cls == 1))
tn_sklearn = np.sum((y_test_cls == 0) & (y_pred_cls == 0))
fp_sklearn = np.sum((y_test_cls == 0) & (y_pred_cls == 1))
fn_sklearn = np.sum((y_test_cls == 1) & (y_pred_cls == 0))

accuracy_sklearn = (tp_sklearn + tn_sklearn) / len(y_test_cls)
precision_sklearn = tp_sklearn / (tp_sklearn + fp_sklearn) if (tp_sklearn + fp_sklearn) > 0 else 0
recall_sklearn = tp_sklearn / (tp_sklearn + fn_sklearn) if (tp_sklearn + fn_sklearn) > 0 else 0
f1_sklearn = 2 * precision_sklearn * recall_sklearn / (precision_sklearn + recall_sklearn) if (precision_sklearn + recall_sklearn) > 0 else 0

classification_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression (Scikit-learn)",
        "Accuracy": accuracy_sklearn,
        "Precision": precision_sklearn,
        "Recall": recall_sklearn,
        "F1-Score": f1_sklearn,
        "Training Time (s)": classification_train_time,
        "Prediction Time (s)": classification_prediction_time
    },
    {
        "Model": "Logistic Regression (Manual)",
        "Accuracy": accuracy_cls_manual,
        "Precision": precision_cls_manual,
        "Recall": recall_cls_manual,
        "F1-Score": f1_cls_manual,
        "Training Time (s)": cls_train_time_manual,
        "Prediction Time (s)": cls_prediction_time_manual
    }
])

display(classification_comparison)

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Logistic Regression (Scikit-learn),0.775000,0.780822,0.966102,0.863636,0.042075,0.011689
1,Logistic Regression (Manual),0.770833,0.779817,0.960452,0.860759,0.120343,0.000437


In [30]:
regression_comparison["Task"] = "Regression"
classification_comparison["Task"] = "Classification"

combined_comparison = pd.concat(
    [regression_comparison, classification_comparison],
    ignore_index=True,
    sort=False
)

display(combined_comparison)

,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s),Task,Accuracy,Precision,Recall,F1-Score
0,Linear Regression (Scikit-learn),0.108453,0.148618,0.168168,0.086498,0.008395,Regression,NaN,NaN,NaN,NaN
1,Linear Regression (Manual),0.108457,0.148628,0.168047,0.009659,0.000544,Regression,NaN,NaN,NaN,NaN
2,Logistic Regression (Scikit-learn),NaN,NaN,NaN,0.042075,0.011689,Classification,0.775000,0.780822,0.966102,0.863636
3,Logistic Regression (Manual),NaN,NaN,NaN,0.120343,0.000437,Classification,0.770833,0.779817,0.960452,0.860759


In [31]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train_cls_manual))

split_point = int(0.8 * len(indices))

train_part = indices[:split_point]
val_part = indices[split_point:]

X_train_opt = X_train_manual[train_part]
y_train_opt = y_train_cls_manual[train_part]

X_val_opt = X_train_manual[val_part]
y_val_opt = y_train_cls_manual[val_part]

best_f1 = -1
best_params = None

for learning_rate in [0.01, 0.05, 0.1, 0.2]:
    for iterations in [1000, 3000, 5000]:
        for l2 in [0.001, 0.01, 0.1]:
            _, y_val_pred, _, _ = manual_logistic_regression(
                X_train_opt,
                y_train_opt,
                X_val_opt,
                learning_rate=learning_rate,
                iterations=iterations,
                l2=l2
            )

            tp = np.sum((y_val_opt == 1) & (y_val_pred == 1))
            fp = np.sum((y_val_opt == 0) & (y_val_pred == 1))
            fn = np.sum((y_val_opt == 1) & (y_val_pred == 0))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

            if f1 > best_f1:
                best_f1 = f1
                best_params = {
                    "learning_rate": learning_rate,
                    "iterations": iterations,
                    "l2": l2
                }

print("Best Parameters:", best_params)
print("Best Validation F1-Score:", best_f1)

Best Parameters: {'learning_rate': 0.01, 'iterations': 1000, 'l2': 0.001}
Best Validation F1-Score: 0.8349514563106797


In [ ]:
start_time = time.perf_counter()

y_train_pred_opt, y_test_pred_opt, y_test_prob_opt, opt_weights = manual_logistic_regression(
    X_train_manual,
    y_train_cls_manual,
    X_test_manual,
    learning_rate=best_params["learning_rate"],
    iterations=best_params["iterations"],
    l2=best_params["l2"]
)

opt_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()

X_test_bias_opt = np.c_[np.ones(X_test_manual.shape[0]), X_test_manual]
y_test_prob_opt = sigmoid(X_test_bias_opt @ opt_weights)
y_test_pred_opt = (y_test_prob_opt >= 0.5).astype(int)

opt_prediction_time = time.perf_counter() - start_time

print("Optimized Model Training Time:", opt_train_time)
print("Optimized Model Prediction Time:", opt_prediction_time)

In [32]:
start_time = time.perf_counter()

y_train_pred_opt, y_test_pred_opt, y_test_prob_opt, opt_weights = manual_logistic_regression(
    X_train_manual,
    y_train_cls_manual,
    X_test_manual,
    learning_rate=best_params["learning_rate"],
    iterations=best_params["iterations"],
    l2=best_params["l2"]
)

opt_train_time = time.perf_counter() - start_time

start_time = time.perf_counter()

X_test_bias_opt = np.c_[np.ones(X_test_manual.shape[0]), X_test_manual]
y_test_prob_opt = sigmoid(X_test_bias_opt @ opt_weights)
y_test_pred_opt = (y_test_prob_opt >= 0.5).astype(int)

opt_prediction_time = time.perf_counter() - start_time

print("Optimized Model Training Time:", opt_train_time)
print("Optimized Model Prediction Time:", opt_prediction_time)

Optimized Model Training Time: 0.05540149999433197
Optimized Model Prediction Time: 0.0004075000179000199


In [33]:
tp_opt = np.sum((y_test_cls_manual == 1) & (y_test_pred_opt == 1))
tn_opt = np.sum((y_test_cls_manual == 0) & (y_test_pred_opt == 0))
fp_opt = np.sum((y_test_cls_manual == 0) & (y_test_pred_opt == 1))
fn_opt = np.sum((y_test_cls_manual == 1) & (y_test_pred_opt == 0))

accuracy_opt = (tp_opt + tn_opt) / len(y_test_cls_manual)
precision_opt = tp_opt / (tp_opt + fp_opt) if (tp_opt + fp_opt) > 0 else 0
recall_opt = tp_opt / (tp_opt + fn_opt) if (tp_opt + fn_opt) > 0 else 0
f1_opt = (
    2 * precision_opt * recall_opt / (precision_opt + recall_opt)
    if (precision_opt + recall_opt) > 0 else 0
)

optimized_results = pd.DataFrame([{
    "Model": "Logistic Regression (Optimized Manual)",
    "Accuracy": accuracy_opt,
    "Precision": precision_opt,
    "Recall": recall_opt,
    "F1-Score": f1_opt,
    "Training Time (s)": opt_train_time,
    "Prediction Time (s)": opt_prediction_time
}])

display(optimized_results)

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s)
0,Logistic Regression (Optimized Manual),0.745833,0.761261,0.954802,0.847118,0.055401,0.000408


In [34]:
final_classification_comparison = pd.concat(
    [
        classification_comparison,
        optimized_results
    ],
    ignore_index=True
)

display(final_classification_comparison)

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s),Task
0,Logistic Regression (Scikit-learn),0.775000,0.780822,0.966102,0.863636,0.042075,0.011689,Classification
1,Logistic Regression (Manual),0.770833,0.779817,0.960452,0.860759,0.120343,0.000437,Classification
2,Logistic Regression (Optimized Manual),0.745833,0.761261,0.954802,0.847118,0.055401,0.000408,NaN


In [35]:
print("FINAL REGRESSION COMPARISON")
display(regression_comparison)

print("FINAL CLASSIFICATION COMPARISON")
display(final_classification_comparison)

FINAL REGRESSION COMPARISON


,Model,MAE,RMSE,R2,Training Time (s),Prediction Time (s),Task
0,Linear Regression (Scikit-learn),0.108453,0.148618,0.168168,0.086498,0.008395,Regression
1,Linear Regression (Manual),0.108457,0.148628,0.168047,0.009659,0.000544,Regression


FINAL CLASSIFICATION COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s),Task
0,Logistic Regression (Scikit-learn),0.775000,0.780822,0.966102,0.863636,0.042075,0.011689,Classification
1,Logistic Regression (Manual),0.770833,0.779817,0.960452,0.860759,0.120343,0.000437,Classification
2,Logistic Regression (Optimized Manual),0.745833,0.761261,0.954802,0.847118,0.055401,0.000408,NaN


In [36]:
optimized_results["Task"] = "Classification"

final_classification_comparison = pd.concat(
    [classification_comparison, optimized_results],
    ignore_index=True
)

display(final_classification_comparison)

,Model,Accuracy,Precision,Recall,F1-Score,Training Time (s),Prediction Time (s),Task
0,Logistic Regression (Scikit-learn),0.775000,0.780822,0.966102,0.863636,0.042075,0.011689,Classification
1,Logistic Regression (Manual),0.770833,0.779817,0.960452,0.860759,0.120343,0.000437,Classification
2,Logistic Regression (Optimized Manual),0.745833,0.761261,0.954802,0.847118,0.055401,0.000408,Classification


In [37]:
regression_comparison.to_csv("regression_comparison.csv", index=False)
final_classification_comparison.to_csv("classification_comparison.csv", index=False)

print("Comparison tables saved successfully.")

Comparison tables saved successfully.


## Final Observations

### Regression
- The manual Linear Regression model achieved results close to the Scikit-learn model.
- Both models produced similar MAE, RMSE, and R² scores.
- The manual implementation had lower training and prediction times in this run.

### Classification
- The manual Logistic Regression model achieved performance close to the Scikit-learn model.
- The optimized manual model achieved a higher recall but lower accuracy than the other classification models.
- Training and prediction times varied across implementations.

### Conclusion
Both manual implementations produced results comparable to their Scikit-learn counterparts. The experiment demonstrates how regression and classification workflows can be implemented using NumPy and Pandas.